# 00 — Setup | Catalog, Schemas and Volumes

**Project:** MLB Performance Intelligence Platform

Creates the Unity Catalog foundation. Run once; every statement is idempotent.

| Object | Purpose |
|---|---|
| Catalog `mlb_intelligence` | Isolates the project from other workspaces objects |
| Schema `bronze` | Raw MLB Stats API responses, stored as received |
| Schema `silver` | Cleaned dimensions (teams, players, venues) and facts (games, team games, batting, pitching) |
| Schema `gold` | Business KPIs and the star-schema serving layer for Power BI |
| Schema `metadata` | Run log, extraction register, incremental watermarks, streaming checkpoints |
| Schema `quality` | Data-quality check results |
| Volume `bronze.mlb_api_raw` | Append-only landing archive: one JSON file per API request |
| Volume `gold.exports` | JSON export that powers the static web platform |
| Volume `metadata.checkpoints` | Auto Loader checkpoints (kept outside the landing archive) |

In [ ]:
import sys, importlib
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + _nb.rsplit("/", 2)[0] + "/shared")
import config as C
importlib.reload(C)

In [ ]:
spark.sql(f"""
CREATE CATALOG IF NOT EXISTS {C.CATALOG}
COMMENT 'MLB Performance Intelligence Platform: games, teams and player performance from the MLB Stats API (2024 onward)'
""")

schemas = {
    "bronze":   "Raw MLB Stats API responses, one row per extracted file, payload kept as received",
    "silver":   "Cleaned, typed and deduplicated dimensions (teams, players, venues) and facts (games, team games, batting, pitching)",
    "gold":     "Business KPIs (team, player, head-to-head, Dominican players) and the star-schema serving layer for BI",
    "metadata": "Pipeline run log, extraction register, incremental watermarks and streaming checkpoints",
    "quality":  "Data-quality check results per run",
}
for schema, comment in schemas.items():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {C.CATALOG}.{schema} COMMENT '{comment}'")

spark.sql(f"""CREATE VOLUME IF NOT EXISTS {C.CATALOG}.bronze.mlb_api_raw
COMMENT 'Append-only landing archive: one JSON envelope per MLB Stats API request, organized as <endpoint>/<yyyy>/<mm>/'""")
spark.sql(f"""CREATE VOLUME IF NOT EXISTS {C.CATALOG}.gold.exports
COMMENT 'Exports of Gold data for the static web platform (mlb_web_data.json)'""")
spark.sql(f"""CREATE VOLUME IF NOT EXISTS {C.CATALOG}.metadata.checkpoints
COMMENT 'Auto Loader / Structured Streaming checkpoints'""")

In [ ]:
display(spark.sql(f"SHOW SCHEMAS IN {C.CATALOG}"))
display(spark.sql(f"SHOW VOLUMES IN {C.CATALOG}.bronze"))